In [4]:
%pip install --no-cache-dir pdfplumber sentence_transformers

Note: you may need to restart the kernel to use updated packages.


In [5]:
import re
import pdfplumber

In [6]:
PDF_PATH = "data/constitution_eng.pdf"
FIRST_PAGE = 33
LAST_PAGE = 283

Remove Footnotes

In [7]:
def crop_above_footnotes(page):
    # pdfplumer has different attributes for lines and rects, so we need to filter out the lines that are actually separators
    separators = [l for l in page.lines if abs(l["top"] - l["bottom"]) < 1 and (l["x1"] - l["x0"]) > 100] # a line could be horizontal or vertical, this line checks that
    if not separators:
        return page
    cut_y = min(l["top"] for l in separators)
    return page.crop((0, 0, page.width, cut_y))


In [8]:
def drop_small_chars(page, min_size=7):
    return page.filter(lambda obj: obj.get("object_type") != "char" or obj["size"] >= min_size)

In [9]:
HEADER_RE = re.compile(r"^\d*\s*THE CONSTITUTION OF INDIA$"   # running title
                       r"|^\(Part [^)]*\)$"                    # (Part III.—...)
                       r"|^\d+$")                              # bare page number

def clean_page(page):
    page = crop_above_footnotes(page)
    page = drop_small_chars(page)
    text = page.extract_text() or ""
    return "\n".join(ln for ln in text.splitlines()
                     if not HEADER_RE.match(ln.strip()))

In [10]:
with pdfplumber.open(PDF_PATH) as pdf:
    page_texts = [clean_page(pdf.pages[i]) for i in range(FIRST_PAGE - 1, LAST_PAGE)]

articles_text = "\n".join(page_texts)
print(len(articles_text))

492608


Saving/Reading the articles

In [11]:
import os

CACHE = "data/articles_text.txt"

if os.path.exists(CACHE): # agar already exust karta hai, then read
    with open(CACHE, encoding="utf-8") as f:
        articles_text = f.read()
else:
    with pdfplumber.open(PDF_PATH) as pdf: # for the first time, extract and cache
        articles_text = "\n".join(clean_page(pdf.pages[i]) for i in range(FIRST_PAGE - 1, LAST_PAGE))
    with open(CACHE, "w", encoding="utf-8") as f:
        f.write(articles_text)

Some preprocessing

The idea is to separate the title amd the body

In [12]:
ARTICLE_RE = re.compile(
    r"^\[?(\d{1,3}[A-Z]{0,3})\.\s+"        # "19. " or "[21A. " at the start of a line
    r"([^—⎯\n]+(?:\n[^—⎯\n]+){0,2}?)"      # title: up to 3 lines, no dash inside
    r"\s*[—⎯]",                             # the title ends at an em-dash
    re.M,
)

In [13]:
matches = list(ARTICLE_RE.finditer(articles_text))
matches


[<re.Match object; span=(35, 71), match='1. Name and territory of the Union.—'>,
 <re.Match object; span=(406, 451), match='2. Admission or establishment of new States.—'>,
 <re.Match object; span=(567, 615), match='[2A. [Sikkim to be associated with the Union.].—'>,
 <re.Match object; span=(705, 797), match='3. Formation of new States and alteration of area>,
 <re.Match object; span=(2026, 2185), match='4. Laws made under articles 2 and 3 to provide fo>,
 <re.Match object; span=(2786, 2842), match='5. Citizenship at the commencement of the Constit>,
 <re.Match object; span=(3225, 3311), match='6. Rights of citizenship of certain persons who h>,
 <re.Match object; span=(4429, 4487), match='7. Rights of citizenship of certain migrants to P>,
 <re.Match object; span=(5147, 5232), match='8. Rights of citizenship of certain persons of In>,
 <re.Match object; span=(5960, 6044), match='9. Persons voluntarily acquiring citizenship of a>,
 <re.Match object; span=(6250, 6296), match='10. Contin

In [14]:
matches[0], matches[0].start(), matches[0].group(1), matches[0].group(2)

(<re.Match object; span=(35, 71), match='1. Name and territory of the Union.—'>,
 35,
 '1',
 'Name and territory of the Union.')

35 = matches[0].start(), the index of the first matched character (the 1 of 1. Name...). </br>
71 = matches[0].end(), the index just after the last matched character (one past the —).

In [15]:
END_PUNCT = (".", ";", ":", "—", "⎯", "]", ",", "-", ")")

def split_trailing_headings(body):
    lines = body.splitlines()
    headings = []
    while lines and not lines[-1].rstrip().endswith(END_PUNCT):
        headings.insert(0, lines.pop().strip())   # pop from the end, keep original order
    return "\n".join(lines), headings

In [16]:
articles = []
for i, m in enumerate(matches):
    end = matches[i + 1].start() if i + 1 < len(matches) else len(articles_text)
    body, trailing = split_trailing_headings(articles_text[m.end():end].strip())

    articles.append({
        "number": m.group(1),
        "title":  " ".join(m.group(2).split()).rstrip("."),   # join the wrapped lines
        "body":  body,
    })

print(len(articles))

500


In [17]:
articles[3]

{'number': '3',
 'title': 'Formation of new States and alteration of areas, boundaries or names of existing States',
 'body': 'Parliament may by law—\n(a) form a new State by separation of territory from any\nState or by uniting two or more States or parts of States or by\nuniting any territory to a part of any State;\n(b) increase the area of any State;\n(c) diminish the area of any State;\n(d) alter the boundaries of any State;\n(e) alter the name of any State:\n[Provided that no Bill for the purpose shall be introduced in\neither House of Parliament except on the recommendation of the\nPresident and unless, where the proposal contained in the Bill affects the\narea, boundaries or name of any of the States ***, the Bill has been\nreferred by the President to the Legislature of that State for expressing\nits views thereon within such period as may be specified in the reference\nor within such further period as the President may allow and the period\nso specified or allowed has expired

Some known issues

In [18]:
%%script false --no-raise-error
articles[4] # before - check the last wala line

Couldn't find program: 'false'


Article 4 chunk ends with the start of the next part (\nPART II\nCITIZENSHIP'). This can affect the ranking of chunking. </br>

For Article 4 the harm is small: "Citizenship" is one word in an article about altering States, so it pulls the embedding only slightly. It matters more where the heading is longer or closer in meaning, like Article 11 picking up "FUNDAMENTAL RIGHTS".

In [19]:
articles[4] # now fixed

{'number': '4',
 'title': 'Laws made under articles 2 and 3 to provide for the amendment of the First and the Fourth Schedules and supplemental, incidental and consequential matters',
 'body': '(1) Any law referred to in\narticle 2 or article 3 shall contain such provisions for the amendment of\nthe First Schedule and the Fourth Schedule as may be necessary to give\neffect to the provisions of the law and may also contain such\nsupplemental, incidental and consequential provisions (including\nprovisions as to representation in Parliament and in the Legislature or\nLegislatures of the State or States affected by such law) as Parliament\nmay deem necessary.\n(2) No such law as aforesaid shall be deemed to be an amendment\nof this Constitution for the purposes of article 368.'}

Chunking


                    Article body
                         │
          ≤ 1000 chars? ─┼─ yes → 1 chunk (whole article)
                         │
                         no
                         ▼
          split at numbered clauses (1), (2), (3)…
                         │
          each clause ≤ 1000? ─┼─ yes → 1 chunk per clause
                               │
                               no
                               ▼
          split at line boundaries into ~1000-char windows
                         │
                         ▼
     every chunk gets "Article N — title" put in front


In [20]:
MAX_CHARS = 1000   # ~256 tokens, the bi-encoder's input limit

# split *before* a line starting with a numbered clause: (1), (2), (1A), [(3) ...
CLAUSE_RE = re.compile(r"^(?=\[?\(\d+[A-Z]?\)\s)", re.M)

In [21]:
def window_split(text, max_chars=MAX_CHARS):
    windows, buf = [], ""
    for line in text.splitlines():
        if buf and len(buf) + len(line) + 1 > max_chars:   # adding this line would overflow
            windows.append(buf)                            # close the current window
            buf = line                                     # start a new one with this line
        else:
            buf = buf + "\n" + line if buf else line
    if buf:
        windows.append(buf)                                # don't lose the last window
    return windows

In [22]:
chunks = []
for a in articles:
    prefix = f"Article {a['number']} — {a['title']}\n"

    # level 1: short article stays whole, long one splits at clauses
    if len(a["body"]) <= MAX_CHARS:
        pieces = [a["body"]]
    else:
        pieces = [p.strip() for p in CLAUSE_RE.split(a["body"]) if p.strip()]

    # level 2: any clause still too long gets windowed by lines
    final = []
    for p in pieces:
        final.extend(window_split(p) if len(p) > MAX_CHARS else [p])

    for j, piece in enumerate(final):
        chunks.append({
            "chunk_id": f"{a['number']}_{j}",   # "19_2" = Article 19, piece 2
            "article":  a["number"],
            "text":     prefix + piece,         # every chunk knows its article
        })

print(len(chunks))


1035


In [23]:
lens = [len(c["text"]) for c in chunks]
max(lens), min(lens), sum(lens) / len(lens)

(1124, 53, 509.57294685990337)

In [24]:
[{c["chunk_id"], c["article"], c["text"]} for c in chunks if c["article"] == "19"]   # 19_0 ... 19_5

[{'19',
  '19_0',
  'Article 19 — Protection of certain rights regarding freedom of speech, etc\n(1) All citizens shall have the right—\n(a) to freedom of speech and expression;\n(b) to assemble peaceably and without arms;\n(c) to form associations or unions [or co-operative societies];\n(d) to move freely throughout the territory of India;\n(e) to reside and settle in any part of the territory of India; [and]\n[(f)* * * * *]\n(g) to practise any profession, or to carry on any occupation, trade or\nbusiness.'},
 {'19',
  '19_1',
  'Article 19 — Protection of certain rights regarding freedom of speech, etc\n[(2) Nothing in sub-clause (a) of clause (1) shall affect the operation of\nany existing law, or prevent the State from making any law, in so far as such law\nimposes reasonable restrictions on the exercise of the right conferred by the said\nsub-clause in the interests of [the sovereignty and integrity of India], the security\nof the State, friendly relations with foreign States, pu

Embedding

In [25]:
from sentence_transformers import SentenceTransformer
import numpy as np

In [26]:
bi_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print(bi_encoder.max_seq_length)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4737.14it/s]


256


In [27]:
chunk_texts = [c["text"] for c in chunks]
chunk_embs = bi_encoder.encode(
    chunk_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,    # unit length, so dot product == cosine similarity
)
print(chunk_embs.shape)

Batches: 100%|██████████| 17/17 [00:28<00:00,  1.65s/it]

(1035, 384)


Retrieval

In [28]:
import time

In [29]:
def retrieve(query, k=10):
    t0 = time.perf_counter()
    q = bi_encoder.encode(query, normalize_embeddings=True) # same model, same normalization
    scores = chunk_embs @ q # (1035,) cosine scores
    top_idx = np.argsort(-scores)[:k] # indices of the k highest
    elapsed = time.perf_counter() - t0

    results = [{"rank": r + 1,
                "chunk_id": chunks[i]["chunk_id"],
                "score": float(scores[i]),
                "text": chunks[i]["text"]}
               for r, i in enumerate(top_idx)]
    return results, elapsed

In [30]:
from sentence_transformers import CrossEncoder

In [32]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, candidates, k=3):
    t0 = time.perf_counter()
    pairs = [(query, c["text"]) for c in candidates] # 10 (query, chunk) pairs
    ce_scores = cross_encoder.predict(pairs) # one score per pair
    elapsed = time.perf_counter() - t0

    reranked = [dict(c, ce_score=float(s)) for c, s in zip(candidates, ce_scores)]
    reranked.sort(key=lambda c: c["ce_score"], reverse=True)
    for new_rank, c in enumerate(reranked, 1):
        c["new_rank"] = new_rank # c["rank"] still holds the bi-encoder rank
    return reranked[:k], reranked, elapsed

c:\Users\Satyajit\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Satyajit\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3065.63it/s]


In [34]:
query = "Can the State make special laws only for women?"

top10, t_bi = retrieve(query)
top3_rr, all_rr, t_ce = rerank(query, top10)

top10

[{'rank': 1,
  'chunk_id': '15_2',
  'score': 0.559055745601654,
  'text': 'Article 15 — Prohibition of discrimination on grounds of religion, race, caste, sex or place of birth\n(3) Nothing in this article shall prevent the State from making any special\nprovision for women and children.'},
 {'rank': 2,
  'chunk_id': '246_2',
  'score': 0.5354660749435425,
  'text': 'Article 246 — Subject-matter of laws made by Parliament and by the Legislatures of States\n(3) Subject to clauses (1) and (2), the Legislature of any State *** has\nexclusive power to make laws for such State or any part thereof with respect to\nany of the matters enumerated in List II in the Seventh Schedule (in this\nConstitution referred to as the “State List”).'},
 {'rank': 3,
  'chunk_id': '15_3',
  'score': 0.4915139973163605,
  'text': 'Article 15 — Prohibition of discrimination on grounds of religion, race, caste, sex or place of birth\n[(4) Nothing in this article or in clause (2) of article 29 shall prevent the\

In [35]:
top3_rr

[{'rank': 1,
  'chunk_id': '15_2',
  'score': 0.559055745601654,
  'text': 'Article 15 — Prohibition of discrimination on grounds of religion, race, caste, sex or place of birth\n(3) Nothing in this article shall prevent the State from making any special\nprovision for women and children.',
  'ce_score': 6.14552640914917,
  'new_rank': 1},
 {'rank': 9,
  'chunk_id': '15_4',
  'score': 0.4567544758319855,
  'text': 'Article 15 — Prohibition of discrimination on grounds of religion, race, caste, sex or place of birth\n[(5) Nothing in this article or in sub-clause (g) of clause (1) of article 19\nshall prevent the State from making any special provision, by law, for the\nadvancement of any socially and educationally backward classes of citizens or\nfor the Scheduled Castes or the Scheduled Tribes in so far as such special\nprovisions relate to their admission to educational institutions including private\neducational institutions, whether aided or unaided by the State, other than the\nmin

In [36]:
for c in all_rr:
    print(f"bi #{c['rank']:>2} -> ce #{c['new_rank']:>2}  {c['chunk_id']:<8} ce={c['ce_score']:6.2f}")

bi # 1 -> ce # 1  15_2     ce=  6.15
bi # 9 -> ce # 2  15_4     ce= -1.52
bi # 3 -> ce # 3  15_3     ce= -2.03
bi # 8 -> ce # 4  332A_0   ce= -2.47
bi # 7 -> ce # 5  15_0     ce= -7.69
bi # 2 -> ce # 6  246_2    ce= -8.47
bi # 4 -> ce # 7  246_0    ce= -9.42
bi # 5 -> ce # 8  304_0    ce=-10.39
bi # 6 -> ce # 9  32A_0    ce=-10.39
bi #10 -> ce #10  187_0    ce=-10.73


## Calling an LLM

In [37]:
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
client = genai.Client()

for m in client.models.list():
    if "flash" in m.name:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview


In [39]:
GEMINI_MODEL = "gemini-3.5-flash"

def generate(prompt):
    t0 = time.perf_counter()
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0),
    )
    return response.text, time.perf_counter() - t0

In [42]:
PROMPT_TEMPLATE = """You are answering questions about the Constitution of India.
Use ONLY the context below. If the context does not contain the answer, say
"The provided context does not answer this." Cite article numbers for every claim.

Context:
{context}

Question: {question}

Answer:"""

def build_prompt(question, top3):
    context = "\n\n".join(f"[{c['chunk_id']}]\n{c['text']}" for c in top3)
    return PROMPT_TEMPLATE.format(context=context, question=question)

In [43]:
query = "Can the State make special laws only for women?"

top10, t_bi = retrieve(query)
top3_rr, all_rr, t_ce = rerank(query, top10)

In [45]:
answer1, t_llm1 = generate(build_prompt(query, top10[:3])) # Pipeline 1: bi-encoder top 3
answer2, t_llm2 = generate(build_prompt(query, top3_rr)) # Pipeline 2: re-ranked top 3

print("ANSWER 1 (baseline):\n", answer1, "\n")
print("ANSWER 2 (re-ranked):\n", answer2)

ANSWER 1 (baseline):
 Based on the provided context, yes, the State is not prevented from making special provisions for women. Article 15(3) states that nothing in Article 15 shall prevent the State from making any special provision for women and children. 

ANSWER 2 (re-ranked):
 Yes. According to Article 15(3), nothing in Article 15 shall prevent the State from making any special provision for women and children.


## Failure Cases